# 1. The gauge wave: a first numerical-relativity run

Flat spacetime in a wiggly time slicing,
$$ds^2 = -H\,dt^2 + H\,dx^2 + dy^2 + dz^2,\qquad H = 1 - A\sin\frac{2\pi(x-t)}{d}.$$
Space-time is flat, but the ADM variables oscillate. Because we know the exact solution, we can measure the error
and check that it converges at 4th order. This is the *Apples-with-Apples* gauge-wave test.

## Settings: where runs are written and read

This notebook writes runs to, and reads them from, **`OUTPUT_ROOT`**. By default
that is `<repository>/simulations`, the same place `pynr run par/<name>.par`
writes to from the shell. So you can:

* run here (`RUN_SIMULATIONS = True`), or
* run in a terminal, e.g. `pynr run par/gauge_wave.par`, then set
  `RUN_SIMULATIONS = False` and only plot.

Change `OUTPUT_ROOT` to keep this notebook's runs elsewhere. For the shell to
use the same folder, run `PYNR_OUTPUT_DIR=<that folder> pynr run ...` (or
`pynr run ... --output-root <that folder>`).

In [ ]:
import os
from pynr import paths

OUTPUT_ROOT = paths.output_root()                   # default: <repo>/simulations (same as `pynr run`)
# OUTPUT_ROOT = os.path.expanduser("~/pynr_runs")   # ...or any folder you like
RUN_SIMULATIONS = True                              # False: only plot runs that already exist

os.environ["PYNR_OUTPUT_DIR"] = OUTPUT_ROOT         # every Simulation below writes here
print("output root  :", OUTPUT_ROOT)
print("existing runs:", paths.list_runs())

## Run (optional)

`gauge_wave` is the parameter file exactly as shipped (10 crossing times, $\Delta x = 0.02$), the same run as
`pynr run par/gauge_wave.par`. The three `gauge_wave_dx*` runs go to $t = 1$ for the convergence test, and write
the lapse at their last iteration.

In [ ]:
from pynr import Simulation

PAR = "../par/gauge_wave.par"
CONVERGENCE = (0.04, 0.02, 0.01)

def convergence_overrides(dx, t_final=1.0):
    n_iter = round(t_final / (0.25 * dx))            # dt = 0.25 dx
    return {"CoordBase::dx": dx, "CoordBase::dy": dx, "CoordBase::dz": dx,
            "CoordBase::ymin": -dx, "CoordBase::ymax": dx, "CoordBase::zmin": -dx, "CoordBase::zmax": dx,
            "Cactus::cctk_final_time": t_final, "IOBasic::outInfo_every": 0,
            "IOHDF5::out2D_every": n_iter, "IOHDF5::out2D_vars": "ADMBase::alp",
            "IO::out_dir": f"gauge_wave_dx{dx}"}

if RUN_SIMULATIONS:
    Simulation.from_parfile(PAR, overrides={"IOBasic::outInfo_every": 0}, verbose=False).run()
    for dx in CONVERGENCE:
        Simulation.from_parfile(PAR, overrides=convergence_overrides(dx), verbose=False).run()
print("runs now:", paths.list_runs())

## Plot: lapse against the exact solution

Everything below reads the output files only (through kuibit), so it works the same for runs started in a terminal.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from kuibit.simdir import SimDir
from pynr.cactus.parfile import parse_parfile
from pynr.thorns.exact import gauge_wave

def lapse_line(run):
    """alpha(x) on the y = 0 line at the last 2D output, and its exact value."""
    sd = SimDir(paths.run_dir(run))
    alp = sd.gf.xy["alp"]
    u = alp[alp.available_iterations[-1]].get_level(0)
    x = u.x0[0] + u.dx[0] * np.arange(u.data.shape[0])
    p = parse_parfile(os.path.join(paths.run_dir(run), "parameters.par"))
    A, d = p["exact::minkowski_gauge_wave__amplitude"], p["exact::minkowski_gauge_wave__lambda"]
    exact = gauge_wave(u.time, x, 0.0, 0.0, A, d)[2]
    return x, u.data[:, u.data.shape[1] // 2], exact, u.time

x, a, ae, t = lapse_line("gauge_wave")
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(x, a, label="pyNR"); ax[0].plot(x, ae, "--", label="exact"); ax[0].legend()
ax[1].plot(x, a - ae); ax[1].set_ylabel("error"); ax[1].set_xlabel("x")
ax[0].set_title(f"run 'gauge_wave', t = {t:.2f}")

## Convergence

Halving $\Delta x$ should reduce the error by $2^4 = 16$.

In [ ]:
errs = {}
for dx in CONVERGENCE:
    x, a, ae, t = lapse_line(f"gauge_wave_dx{dx}")
    errs[dx] = np.abs(a - ae)[3:-3].max()            # skip the ghost points
    plt.plot(x, (a - ae) * (CONVERGENCE[0] / dx) ** 4, label=f"dx = {dx}, error x (0.04/dx)^4")
    print(f"dx = {dx:5.3f}   t = {t:.3f}   max error = {errs[dx]:.3e}")
plt.legend(); plt.xlabel("x"); plt.title("scaled errors coincide -> 4th order")
for a_, b_ in zip(CONVERGENCE[:-1], CONVERGENCE[1:]):
    print(f"order between dx={a_} and dx={b_}: {np.log2(errs[a_] / errs[b_]):.2f}")

## Time series

The run directory has the same layout as an Einstein Toolkit run, so all of kuibit works.

In [ ]:
sd = SimDir(paths.run_dir("gauge_wave"))
amax, amin = sd.ts.maximum["alp"], sd.ts.minimum["alp"]
plt.plot(amax.t, amax.y, label="max alpha"); plt.plot(amin.t, amin.y, label="min alpha")
plt.axhline(np.sqrt(1.1), ls=":", c="k"); plt.axhline(np.sqrt(0.9), ls=":", c="k")
plt.xlabel("t"); plt.legend()

### Exercises
1. Increase the amplitude to $A = 0.5$. How long does the run survive? (The gauge wave is famous for exposing instabilities in formulations.)
2. Switch `MoL::ODE_Method` to `"RK2"` and redo the convergence test. What order do you measure, and why?
3. Set `ADMBase::lapse_evolution_method = "1+log"`. Is the exact solution still a solution?